# 05. Export (Tableau용 데이터 정리)
### 지도학습 + 비지도학습 결과를 하나의 CSV로 통합
---

## 0. 라이브러리 불러오기

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로딩 완료 ✅')

라이브러리 로딩 완료 ✅


---
## 1. 결과 파일 불러오기

In [20]:
# 03번 지도학습 결과 (index_col=0 : 원본 행 번호 복원)
df_supervised = pd.read_csv('../output/supervised_results.csv', index_col=0, encoding='utf-8-sig')

# 04번 비지도학습 결과 (index_col=0 : 원본 행 번호 복원)
df_unsupervised = pd.read_csv('../output/unsupervised_results.csv', index_col=0, encoding='utf-8-sig')

# 04번 t-SNE 결과
df_tsne = pd.read_csv('../output/tsne_results.csv', encoding='utf-8-sig')

print('파일 로딩 완료 ✅')
print(f'지도학습 결과   : {df_supervised.shape}')
print(f'비지도학습 결과 : {df_unsupervised.shape}')
print(f't-SNE 결과      : {df_tsne.shape}')

파일 로딩 완료 ✅
지도학습 결과   : (39835, 13)
비지도학습 결과 : (39835, 9)
t-SNE 결과      : (3000, 9)


In [21]:
# 각 파일 컬럼 확인
print('=== 지도학습 컬럼 ===')
print(df_supervised.columns.tolist())
print()
print('=== 비지도학습 컬럼 ===')
print(df_unsupervised.columns.tolist())
print()
print('=== t-SNE 컬럼 ===')
print(df_tsne.columns.tolist())

=== 지도학습 컬럼 ===
['score', 'content', 'label', 'content_len', 'content_clean', 'content_token', 'pred_logistic', 'pred_rf', 'pred_xgb', 'pred_score_lr', 'correct_logistic', 'correct_rf', 'correct_xgb']

=== 비지도학습 컬럼 ===
['score', 'content', 'label', 'content_len', 'content_clean', 'content_token', 'cluster', 'pca_x', 'pca_y']

=== t-SNE 컬럼 ===
['score', 'content', 'label', 'content_len', 'content_clean', 'content_token', 'cluster', 'tsne_x', 'tsne_y']


---
## 2. 지도학습 + 비지도학습 결과 통합

In [22]:
df_unsup_cols = df_unsupervised[['cluster', 'pca_x', 'pca_y']].copy()

df_sup_cols = df_supervised[[
    'content', 'score', 'label',
    'pred_logistic', 'pred_rf', 'pred_xgb', 'pred_score_lr',
    'correct_logistic', 'correct_rf', 'correct_xgb'
]].copy()

print('컬럼 추출 완료 ✅')
print(f'지도학습   : {df_sup_cols.shape}')
print(f'비지도학습 : {df_unsup_cols.shape}')

컬럼 추출 완료 ✅
지도학습   : (39835, 10)
비지도학습 : (39835, 3)


In [23]:
# 인덱스 기반 join (content 대신 원본 행 번호로 매핑 → NaN 없이 전체 결합)
df_merged = df_sup_cols.join(
    df_unsup_cols[['cluster', 'pca_x', 'pca_y']],
    how='left'
)

print('join 완료 ✅')
print(f'통합 데이터 shape : {df_merged.shape}')
df_merged.head()

join 완료 ✅
통합 데이터 shape : (39835, 13)


,content,score,label,pred_logistic,pred_rf,pred_xgb,pred_score_lr,correct_logistic,correct_rf,correct_xgb,cluster,pca_x,pca_y
49081,"재구매 단맛,짠맛이 적당해서 맛있네요.",5,1,1,1,1,3.774998,1,1,1,1,-0.018195,-0.027485
112278,재구매 불량이 엄청나네요 사탕마다 죄다끈적거리고,1,0,0,0,0,2.137872,1,1,1,3,-0.019508,-0.020016
2770,작동이 잘안됐어요ㅠ,2,0,0,0,0,2.534545,1,1,1,3,-0.029365,-0.023802
45953,재구매 그럭저럭 마음에들고 좋아요.,5,1,1,1,1,5.109655,1,1,1,3,0.059460,0.135057
161037,좋아요 만족,2,0,1,1,1,4.608329,0,0,0,3,0.067388,0.172929


---
## 3. 컬럼 정리 및 가독성 향상
> Tableau에서 보기 편하게 한글 컬럼명 추가


In [24]:
# 예측값 → 긍/부정 텍스트로 변환
label_map = {0: '부정', 1: '긍정'}

df_merged['실제감성']         = df_merged['label'].map(label_map)
df_merged['로지스틱_예측']    = df_merged['pred_logistic'].map(label_map)
df_merged['랜덤포레스트_예측'] = df_merged['pred_rf'].map(label_map)
df_merged['XGBoost_예측']     = df_merged['pred_xgb'].map(label_map)

# 정답여부 → 텍스트로 변환
correct_map = {1: '정답', 0: '오답'}
df_merged['로지스틱_정답여부']    = df_merged['correct_logistic'].map(correct_map)
df_merged['랜덤포레스트_정답여부'] = df_merged['correct_rf'].map(correct_map)
df_merged['XGBoost_정답여부']     = df_merged['correct_xgb'].map(correct_map)

# 별점 텍스트 변환
df_merged['별점텍스트'] = df_merged['score'].astype(str) + '점'

# 군집 텍스트 변환
cluster_map = {
    0: '제품후기형',
    1: '식품카테고리',
    2: '배송중심',
    3: '일반감성형'
}
df_merged['군집명'] = df_merged['cluster'].map(cluster_map)

# 선형회귀 예측 별점 반올림
df_merged['선형회귀_예측별점'] = df_merged['pred_score_lr'].round(2)

print('컬럼 정리 완료 ✅')
df_merged.head(3)

컬럼 정리 완료 ✅


,content,score,label,pred_logistic,pred_rf,pred_xgb,pred_score_lr,correct_logistic,correct_rf,correct_xgb,...,실제감성,로지스틱_예측,랜덤포레스트_예측,XGBoost_예측,로지스틱_정답여부,랜덤포레스트_정답여부,XGBoost_정답여부,별점텍스트,군집명,선형회귀_예측별점
49081,"재구매 단맛,짠맛이 적당해서 맛있네요.",5,1,1,1,1,3.774998,1,1,1,...,긍정,긍정,긍정,긍정,정답,정답,정답,5점,식품카테고리,3.77
112278,재구매 불량이 엄청나네요 사탕마다 죄다끈적거리고,1,0,0,0,0,2.137872,1,1,1,...,부정,부정,부정,부정,정답,정답,정답,1점,일반감성형,2.14
2770,작동이 잘안됐어요ㅠ,2,0,0,0,0,2.534545,1,1,1,...,부정,부정,부정,부정,정답,정답,정답,2점,일반감성형,2.53


---
## 4. 최종 컬럼 선택 및 저장

In [25]:
# Tableau용 최종 컬럼 선택
tableau_cols = [
    # 원본 데이터
    'content',              # 리뷰 텍스트
    'score',                # 실제 별점 (숫자)
    '별점텍스트',            # 실제 별점 (텍스트)
    'label',                # 실제 긍부정 (0/1)
    '실제감성',              # 실제 긍부정 (텍스트)

    # 지도학습 예측
    '로지스틱_예측',
    '랜덤포레스트_예측',
    'XGBoost_예측',
    '선형회귀_예측별점',

    # 정답 여부
    '로지스틱_정답여부',
    '랜덤포레스트_정답여부',
    'XGBoost_정답여부',

    # 비지도학습
    'cluster',              # 군집 번호
    '군집명',               # 군집 이름
    'pca_x',               # PCA 좌표
    'pca_y',
]

df_tableau = df_merged[tableau_cols].copy()

print(f'최종 데이터 shape : {df_tableau.shape}')
print(f'컬럼 수 : {len(df_tableau.columns)}개')
print()
print('=== 최종 컬럼 목록 ===')
for col in df_tableau.columns:
    print(f'  {col}')

최종 데이터 shape : (39835, 16)
컬럼 수 : 16개

=== 최종 컬럼 목록 ===
  content
  score
  별점텍스트
  label
  실제감성
  로지스틱_예측
  랜덤포레스트_예측
  XGBoost_예측
  선형회귀_예측별점
  로지스틱_정답여부
  랜덤포레스트_정답여부
  XGBoost_정답여부
  cluster
  군집명
  pca_x
  pca_y


In [26]:
# t-SNE는 샘플수가 다르므로 별도 저장
df_tsne_export = df_tsne[[
    'content', 'score', 'label', 'cluster', 'tsne_x', 'tsne_y'
]].copy()
df_tsne_export['실제감성'] = df_tsne_export['label'].map(label_map)
df_tsne_export['군집명']   = df_tsne_export['cluster'].map(cluster_map)

# 저장
df_tableau.to_csv('../output/tableau_main.csv',  index=False, encoding='utf-8-sig')
df_tsne_export.to_csv('../output/tableau_tsne.csv', index=False, encoding='utf-8-sig')

print('저장 완료 ✅')
print()
print('저장된 파일 :')
print('  output/tableau_main.csv  ← 메인 데이터 (지도 + 비지도 통합)')
print('  output/tableau_tsne.csv  ← t-SNE 좌표 별도')

저장 완료 ✅

저장된 파일 :
  output/tableau_main.csv  ← 메인 데이터 (지도 + 비지도 통합)
  output/tableau_tsne.csv  ← t-SNE 좌표 별도


---
## 5. 데이터 최종 검증

In [27]:
print('=== 최종 데이터 검증 ===')
print(f'총 데이터 수 : {len(df_tableau):,}개')
print()
print('--- 결측치 확인 ---')
print(df_tableau.isnull().sum())
print()
print('--- 실제감성 분포 ---')
print(df_tableau['실제감성'].value_counts())
print()
print('--- 별점 분포 ---')
print(df_tableau['별점텍스트'].value_counts().sort_index())
print()
print('--- 군집 분포 ---')
print(df_tableau['군집명'].value_counts())

=== 최종 데이터 검증 ===
총 데이터 수 : 39,835개

--- 결측치 확인 ---
content         0
score           0
별점텍스트           0
label           0
실제감성            0
로지스틱_예측         0
랜덤포레스트_예측       0
XGBoost_예측      0
선형회귀_예측별점       0
로지스틱_정답여부       0
랜덤포레스트_정답여부     0
XGBoost_정답여부    0
cluster         0
군집명             0
pca_x           0
pca_y           0
dtype: int64

--- 실제감성 분포 ---
실제감성
긍정    19926
부정    19909
Name: count, dtype: int64

--- 별점 분포 ---
별점텍스트
1점     7203
2점    12706
4점     3807
5점    16119
Name: count, dtype: int64

--- 군집 분포 ---
군집명
일반감성형     25419
제품후기형      9478
식품카테고리     2778
배송중심       2160
Name: count, dtype: int64


In [28]:
# 모델별 정답률 최종 확인
print('=== 모델별 정답률 최종 확인 ===')
for model in ['로지스틱', '랜덤포레스트', 'XGBoost']:
    col = f'{model}_정답여부'
    acc = (df_tableau[col] == '정답').mean() * 100
    print(f'{model:<12} : {acc:.2f}%')

=== 모델별 정답률 최종 확인 ===
로지스틱         : 89.00%
랜덤포레스트       : 84.22%
XGBoost      : 85.45%


In [29]:
# 최종 데이터 샘플 확인
print('=== 최종 데이터 샘플 (5개) ===')
df_tableau.sample(5, random_state=42)

=== 최종 데이터 샘플 (5개) ===


,content,score,별점텍스트,label,실제감성,로지스틱_예측,랜덤포레스트_예측,XGBoost_예측,선형회귀_예측별점,로지스틱_정답여부,랜덤포레스트_정답여부,XGBoost_정답여부,cluster,군집명,pca_x,pca_y
95704,알넣으면 다른 안경이랑 똑같아요 가볍다는거 딱히 모르겠어요 카톡주셔서 답장 드렸는데...,1,1점,0,부정,부정,부정,부정,0.65,정답,정답,정답,3,일반감성형,-0.042610,-0.024774
62251,"반신반의하면서 구입했는데, 돌돌 말 수 있고 마음에 들어요. 아이보리가 있었으면 그...",5,5점,1,긍정,긍정,긍정,긍정,4.90,정답,정답,정답,0,제품후기형,-0.059607,-0.017791
68730,쿨매트라고 하긴 부족하고 그냥 에어매트 정도로 생각하심 될듯요... 더위 타는 아이...,2,2점,0,부정,부정,부정,부정,1.74,정답,정답,정답,3,일반감성형,-0.035660,0.025341
179465,가격도 진짜 저렴한데 발열 장난 아니예요 강추,5,5점,1,긍정,긍정,부정,부정,3.21,정답,오답,오답,3,일반감성형,-0.016806,-0.005721
51111,이미 하나는 버렸고... 하나는 그래도 성공해서 다행... 저는 원래 필름 정말 잘...,2,2점,0,부정,부정,부정,부정,2.30,정답,정답,정답,3,일반감성형,-0.056377,-0.040454


In [41]:
metrics_data = {
    '모델': ['로지스틱회귀']*4 + ['랜덤포레스트']*4 + ['XGBoost']*4,
    '지표': ['Accuracy', 'F1', 'Precision', 'Recall'] * 3,
    '값':  [
        0.8900, 0.8898, 0.8922, 0.8873,  # 로지스틱회귀
        0.8422, 0.8383, 0.8600, 0.8176,  # 랜덤포레스트
        0.8545, 0.8514, 0.8704, 0.8333   # XGBoost
    ]
}

df_metrics = pd.DataFrame(metrics_data)
df_metrics.to_csv('../output/model_metrics.csv', index=False, encoding='utf-8-sig')

print(df_metrics)
print("\n✅ model_metrics.csv 저장 완료")

         모델         지표       값
0    로지스틱회귀   Accuracy  0.8900
1    로지스틱회귀         F1  0.8898
2    로지스틱회귀  Precision  0.8922
3    로지스틱회귀     Recall  0.8873
4    랜덤포레스트   Accuracy  0.8422
5    랜덤포레스트         F1  0.8383
6    랜덤포레스트  Precision  0.8600
7    랜덤포레스트     Recall  0.8176
8   XGBoost   Accuracy  0.8545
9   XGBoost         F1  0.8514
10  XGBoost  Precision  0.8704
11  XGBoost     Recall  0.8333

✅ model_metrics.csv 저장 완료


---
## ✅ 전체 프로젝트 완료 요약

| 단계 | 파일 | 내용 |
|---|---|---|
| 1 | 01_EDA | 데이터 탐색 및 기본 분석 |
| 2 | 02_preprocessing | 형태소 분석 + TF-IDF 변환 |
| 3 | 03_supervised | 선형회귀 / 로지스틱 / 랜덤포레스트 / XGBoost |
| 4 | 04_unsupervised | K-Means / PCA / t-SNE |
| 5 | 05_export | Tableau용 데이터 통합 정리 |

### 최종 산출물
```
output/
├── tableau_main.csv   ← Tableau 메인 데이터
├── tableau_tsne.csv   ← Tableau t-SNE 데이터
└── *.png              ← 분석 결과 이미지
```

---